# Tiny GPT — browser training fixture

This reduced decoder-only transformer exercises TorchLite's `llm-core-v1` API entirely inside the JupyterLite/Pyodide worker. Its deterministic initialization is also executed against real PyTorch in the repository test suite.

In [ ]:
%pip install -q torchlite
import torch
import torch.nn as nn
import torch.nn.functional as F
print('torchlite', torch.__version__)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
    def forward(self, x):
        batch, time, channels = x.shape
        q, k, v = self.c_attn(x).split(channels, dim=2)
        q = q.view(batch, time, self.n_head, channels//self.n_head).transpose(1, 2)
        k = k.view(batch, time, self.n_head, channels//self.n_head).transpose(1, 2)
        v = v.view(batch, time, self.n_head, channels//self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(batch, time, channels)
        return self.c_proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'), nn.Linear(4*n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab=11, context=6, n_embd=8):
        super().__init__()
        self.transformer = nn.ModuleDict({'wte': nn.Embedding(vocab, n_embd), 'wpe': nn.Embedding(context, n_embd), 'blocks': nn.ModuleList([Block(n_embd, 2)]), 'ln_f': nn.LayerNorm(n_embd)})
        self.lm_head = nn.Linear(n_embd, vocab, bias=False)
    def forward(self, tokens, targets=None):
        _, time = tokens.shape
        positions = torch.arange(time, dtype=torch.long, device=tokens.device)
        x = self.transformer['wte'](tokens) + self.transformer['wpe'](positions)
        for block in self.transformer['blocks']: x = block(x)
        logits = self.lm_head(self.transformer['ln_f'](x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

In [ ]:
model = TinyGPT()
for index, parameter in enumerate(model.parameters()):
    values = torch.linspace(-0.04 + index*0.001, 0.04 + index*0.001, parameter.numel(), dtype=parameter.dtype).view(parameter.shape)
    parameter.data.copy_(values)

tokens = torch.tensor([[0,1,2,3,4,5], [5,4,3,2,1,0], [2,3,5,7,1,4], [8,6,4,2,0,9]])
targets = torch.tensor([[1,2,3,4,5,6], [4,3,2,1,0,-1], [3,5,7,1,4,6], [6,4,2,0,9,10]])
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([{'params': decay, 'weight_decay': 0.01}, {'params': no_decay, 'weight_decay': 0.0}], lr=0.025)
losses = []
for step in range(24):
    logits, loss = model(tokens, targets)
    losses.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
logits, loss = model(tokens, targets)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'tiny-gpt loss: {losses[0]:.4f} -> {loss.item():.4f}; parameters={parameter_count}')
assert losses[-1] < losses[0] * 0.72
assert logits.shape == (4, 6, 11)

## What this proves

The full forward/backward/update path executes locally in the browser through the PyTorch-shaped API. This preset is intentionally tiny and NumPy/Wasm-backed; larger models need the planned WebGPU backend.